# Explainable Intrusion Detection for Safety Critical Facilities
### Reducing Operator Alert Fatigue with SHAP Guided Anomaly Scoring

HAI 21.03 (HIL based Augmented ICS Security Dataset). Runtime about 5 minutes on a free Colab CPU runtime.

Pipeline: window the 1 Hz telemetry, detect with a normal only PCA reconstruction detector whose dimensionality
comes from an explained variance criterion and whose threshold comes from a false alarm budget on held out normal
data, attribute the score back to the physical subsystem, then consolidate point alerts into operator facing
incidents using the grouped signature. Neither free parameter is tuned on test data.

## 1. Environment and data

In [ ]:
!pip install -q lightgbm shap
import os
if not os.path.exists('hai'):
    !git clone --depth 1 -q https://github.com/icsdataset/hai.git
# HAI 21.03 is used because it is the version carrying per subsystem labels
# (attack_P1/P2/P3) and because its files are plain gzipped CSV rather than Git LFS stubs.
!ls hai/hai-21.03

In [ ]:
import glob, os, json, time
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
import lightgbm as lgb, shap
t0 = time.time()
OUT = '/content/outputs'; FIG = '/content/figures'
os.makedirs(OUT, exist_ok=True); os.makedirs(FIG, exist_ok=True)
D = 'hai/hai-21.03'; LAB = ['attack','attack_P1','attack_P2','attack_P3']
W, S = 60, 10; VAR = 0.98; BUDGET = 0.5   # window, stride, variance criterion, false alerts per hour
plt.rcParams.update({'savefig.dpi': 300, 'font.size': 9})

## 2. Load and window

Five statistics per sensor (mean, standard deviation, min, max, mean absolute first difference) turn 79 raw
channels into 395 window level features. A window is labelled attack if any second inside it is an attack.

In [ ]:
def load(p):
    df=pd.read_csv(p); df.columns=[c.strip() for c in df.columns]
    df['time']=pd.to_datetime(df['time']); return df.sort_values('time').reset_index(drop=True)

tr=[load(p) for p in sorted(glob.glob(D+'/train*.csv*'))]
te=[load(p) for p in sorted(glob.glob(D+'/test*.csv*'))]
SEN=[c for c in te[0].columns if c not in LAB+['time']]
STATS=['mean','std','min','max','dmean']
FEAT=[f'{s}|{st}' for st in STATS for s in SEN]
GROUPS=np.array([f.split('_')[0] for f in FEAT]); G=['P1','P2','P3','P4']

def wz(df,lab=True):
    V=df[SEN].to_numpy(np.float32); st=np.arange(0,len(V)-W+1,S); F=[]
    for i in range(0,len(st),2000):
        s=st[i:i+2000]; b=V[s[:,None]+np.arange(W)[None,:]]
        F.append(np.concatenate([b.mean(1),b.std(1),b.min(1),b.max(1),np.abs(np.diff(b,1,1)).mean(1)],1).astype(np.float32))
    o={'X':np.vstack(F),'t':df['time'].to_numpy()[st]}
    if lab:
        idx=st[:,None]+np.arange(W)[None,:]
        for L in LAB: o[L]=df[L].to_numpy(np.int8)[idx].max(1)
    return o
Wtr=[wz(d,False) for d in tr]; Wte=[wz(d) for d in te]
Xtr=np.vstack([w['X'] for w in Wtr])
cut=int(0.8*len(Xtr)); Xfit,Xcal=Xtr[:cut],Xtr[cut:]
cal_hours=len(Xcal)*S/3600
print('fit %d  calib %d (%.1f h)  test %d' % (len(Xfit),len(Xcal),cal_hours,sum(len(w['X']) for w in Wte)))

## 3. Detector: normal only, no test derived tuning

The number of components is the smallest k reaching 98 percent of training variance. The threshold is the order
statistic of the calibration scores matching the chosen false alarm budget. Both are set on normal data only.

In [ ]:
sc=StandardScaler().fit(Xfit); Zfit=sc.transform(Xfit); Zcal=sc.transform(Xcal)
p_full=PCA(n_components=150,random_state=0).fit(Zfit)
K=int(np.argmax(np.cumsum(p_full.explained_variance_ratio_)>=VAR))+1
print('components at %.0f%% variance: %d' % (VAR*100,K))
pca=PCA(n_components=K,random_state=0).fit(Zfit)
err=lambda Z: ((Z-pca.inverse_transform(pca.transform(Z)))**2)

def budget_threshold(scores,hours,budget):
    allowed=max(1,int(round(budget*hours)))
    return np.sort(scores)[-allowed]

ecal=err(Zcal).mean(1)
THR=budget_threshold(ecal,cal_hours,BUDGET)
print('threshold at %.1f FA/h = %.5f (calib percentile %.3f)' % (BUDGET,THR,100*(ecal<THR).mean()))

iso=IsolationForest(n_estimators=100,random_state=0,n_jobs=-1).fit(
    Zfit[np.random.RandomState(0).choice(len(Zfit),20000,replace=False)])
THR_ISO=budget_threshold(-iso.score_samples(Zcal),cal_hours,BUDGET)

## 4. Unsupervised detection

In [ ]:
# ---------- unsupervised detection ----------
rows=[];uns=[]
for k,w in enumerate(Wte):
    Z=sc.transform(w['X']); E=err(Z); s=E.mean(1); y=w['attack']
    pred=(s>THR).astype(int)
    p,r,f,_=precision_recall_fscore_support(y,pred,average='binary',zero_division=0)
    si=-iso.score_samples(Z); pi=(si>THR_ISO).astype(int)
    p2,r2,f2,_=precision_recall_fscore_support(y,pi,average='binary',zero_division=0)
    rows.append(dict(file=k+1,n=len(y),attack_rate=y.mean(),pca_P=p,pca_R=r,pca_F1=f,
                     pca_AUC=roc_auc_score(y,s),iso_R=r2,iso_F1=f2,iso_AUC=roc_auc_score(y,si)))
    uns.append(dict(E=E,score=s,pred=pred))
T1=pd.DataFrame(rows); print('\nTABLE 1'); print(T1.round(3).to_string(index=False))
print('means',T1[['pca_P','pca_R','pca_F1','pca_AUC','iso_R','iso_F1','iso_AUC']].mean().round(3).to_dict())

## 5. Sensitivity to the two free choices

In [ ]:
# ---------- ablation ----------
Xall=np.vstack([w['X'] for w in Wte]); yall=np.concatenate([w['attack'] for w in Wte])
Zall=sc.transform(Xall)
test_hours=sum((w['t'].max()-w['t'].min())/np.timedelta64(1,'h') for w in Wte)
ab=[]
cum=np.cumsum(p_full.explained_variance_ratio_)
for v,kk in [(0.90,int(np.argmax(cum>=0.90))+1),(0.95,int(np.argmax(cum>=0.95))+1),
             (0.98,K),(0.99,int(np.argmax(cum>=0.99))+1)]:
    pk=PCA(n_components=kk,random_state=0).fit(Zfit)
    ec=((Zcal-pk.inverse_transform(pk.transform(Zcal)))**2).mean(1)
    ea=((Zall-pk.inverse_transform(pk.transform(Zall)))**2).mean(1)
    auc=roc_auc_score(yall,ea)
    for b in [0.25,0.5,1.0]:
        th=budget_threshold(ec,cal_hours,b); pr=(ea>th).astype(int)
        p,r,f,_=precision_recall_fscore_support(yall,pr,average='binary',zero_division=0)
        ab.append(dict(variance=v,k=kk,budget=b,P=p,R=r,F1=f,AUC=auc,alerts_h=pr.sum()/test_hours))
T5=pd.DataFrame(ab); print('\nTABLE 5 (ablation)'); print(T5.round(3).to_string(index=False))

## 6. Supervised reference

The HAI training files contain no attacks, so the supervised model is trained on four test files and evaluated on
the fifth. This is an upper bound under favourable conditions, not a deployable configuration.

In [ ]:
# ---------- supervised ----------
sup=[];models=[];spred=[]
for held in range(5):
    fit=[i for i in range(5) if i!=held]
    Xf=np.vstack([Wte[i]['X'] for i in fit]); yf=np.concatenate([Wte[i]['attack'] for i in fit])
    Xe,ye=Wte[held]['X'],Wte[held]['attack']
    clf=lgb.LGBMClassifier(n_estimators=300,learning_rate=0.05,num_leaves=63,
                           class_weight='balanced',random_state=0,n_jobs=-1,verbose=-1).fit(Xf,yf)
    pb=clf.predict_proba(Xe)[:,1]; pd_=(pb>0.5).astype(int)
    p,r,f,_=precision_recall_fscore_support(ye,pd_,average='binary',zero_division=0)
    sup.append(dict(fold=held+1,n=len(ye),P=p,R=r,F1=f,AUC=roc_auc_score(ye,pb)))
    models.append(clf); spred.append(pd_)
T2=pd.DataFrame(sup); print('\nTABLE 2'); print(T2.round(3).to_string(index=False))
print('means',T2[['P','R','F1','AUC']].mean().round(3).to_dict())

## 7. Subsystem grouped attribution

ICS sensors are collinear, so attribution credit for one physical event scatters across many channels. Attributions
are summed within a subsystem before ranking. Localisation is scored against attack_P1, P2 and P3.

In [ ]:
# ---------- localization ----------
def loc(att,idx,sub,y):
    m=(sub[idx].sum(1)==1)&(y[idx]==1)
    if m.sum()==0: return None
    A=att[m]; gm=np.stack([A[:,GROUPS==g].sum(1) for g in G],1)
    ti=sub[idx][m].argmax(1); gi=gm[:,:3].argmax(1)
    ni=np.array([G.index(GROUPS[np.argmax(np.abs(A[i]))]) for i in range(len(A))])
    absg=np.abs(gm[:,:3]); conc=(absg.max(1)/(absg.sum(1)+1e-9)).mean()
    return dict(n=int(m.sum()),grouped=float((gi==ti).mean()),naive=float((ni==ti).mean()),
                conc=float(conc),gmat=gm,mask=m)

L=[];store=[]
for held in range(5):
    w=Wte[held]; sub=np.stack([w[f'attack_P{k}'] for k in (1,2,3)],1)
    ds=np.where(spred[held]==1)[0]
    sv=shap.TreeExplainer(models[held]).shap_values(w['X'][ds])
    if isinstance(sv,list): sv=sv[1]
    if np.ndim(sv)==3: sv=sv[:,:,1]
    rs=loc(sv,ds,sub,w['attack'])
    du=np.where(uns[held]['pred']==1)[0]
    ru=loc(uns[held]['E'][du],du,sub,w['attack'])
    L.append(dict(fold=held+1,sup_n=rs['n'],sup_grouped=rs['grouped'],sup_naive=rs['naive'],sup_conc=rs['conc'],
                  un_n=ru['n'],un_grouped=ru['grouped'],un_naive=ru['naive'],un_conc=ru['conc']))
    store.append(dict(sv=sv,idx=ds))
T3=pd.DataFrame(L); print('\nTABLE 3'); print(T3.round(3).to_string(index=False))
print('sup grouped %.3f naive %.3f | un grouped %.3f naive %.3f | conc sup %.3f un %.3f' % (
    np.average(T3.sup_grouped,weights=T3.sup_n),np.average(T3.sup_naive,weights=T3.sup_n),
    np.average(T3.un_grouped,weights=T3.un_n),np.average(T3.un_naive,weights=T3.un_n),
    np.average(T3.sup_conc,weights=T3.sup_n),np.average(T3.un_conc,weights=T3.un_n)))

## 8. Alert consolidation and operator burden

Consecutive alerts merge into one incident when they are close in time and their signatures point at the same
subsystem. The grouped signature is compared against the full 395 dimensional one under the identical rule.

In [ ]:
# ---------- consolidation ----------
def cos(a,b): return float(a@b/(np.linalg.norm(a)*np.linalg.norm(b)+1e-9))
def consolidate(t,sig,gap=300,sim=0.8):
    o=np.argsort(t); t,sg=t[o],sig[o]; n=1
    for i in range(1,len(t)):
        if (t[i]-t[i-1])/np.timedelta64(1,'s')>gap or cos(sg[i],sg[i-1])<sim: n+=1
    return n
def events(y):
    ev=[];st=None
    for i,v in enumerate(y):
        if v and st is None: st=i
        if not v and st is not None: ev.append((st,i-1)); st=None
    if st is not None: ev.append((st,len(y)-1))
    return ev

C=[]
for held in range(5):
    w=Wte[held]; st_=store[held]; idx=st_['idx']; sv=st_['sv']
    gm=np.stack([sv[:,GROUPS==g].sum(1) for g in G],1); tt=w['t'][idx]
    ng=consolidate(tt,gm); nr=consolidate(tt,sv)
    hrs=(w['t'].max()-w['t'].min())/np.timedelta64(1,'h')
    ev=events(w['attack']); fl=set(idx.tolist()); cov=0; lat=[]
    for a,b in ev:
        h=[i for i in range(a,b+1) if i in fl]
        if h: cov+=1; lat.append((w['t'][h[0]]-w['t'][a])/np.timedelta64(1,'s'))
    C.append(dict(fold=held+1,hours=hrs,alerts=len(idx),incidents_grouped=ng,incidents_raw=nr,
                  alerts_per_h=len(idx)/hrs,incidents_per_h=ng/hrs,events=len(ev),
                  events_detected=cov,median_latency_s=float(np.median(lat)) if lat else np.nan))
T4=pd.DataFrame(C); print('\nTABLE 4'); print(T4.round(3).to_string(index=False))
A,I,R=T4.alerts.sum(),T4.incidents_grouped.sum(),T4.incidents_raw.sum()
print('alerts %d -> incidents %d (%.1f%%), raw-signature %d (%.1f%% more), %.1f h, %.2f -> %.2f per h, events %d/%d' % (
    A,I,100*(1-I/A),R,100*(R/I-1),T4.hours.sum(),A/T4.hours.sum(),I/T4.hours.sum(),
    T4.events_detected.sum(),T4.events.sum()))

In [ ]:
# ---------- consolidation on the deployable (unsupervised) path ----------
CU=[]
for held in range(5):
    w=Wte[held]; idx=np.where(uns[held]['pred']==1)[0]
    E=uns[held]['E'][idx]
    gm=np.stack([E[:,GROUPS==g].sum(1) for g in G],1); tt=w['t'][idx]
    ng=consolidate(tt,gm); nr=consolidate(tt,E)
    hrs=(w['t'].max()-w['t'].min())/np.timedelta64(1,'h')
    ev=events(w['attack']); fl=set(idx.tolist()); cov=0; lat=[]
    for a,b in ev:
        h=[i for i in range(a,b+1) if i in fl]
        if h: cov+=1; lat.append((w['t'][h[0]]-w['t'][a])/np.timedelta64(1,'s'))
    CU.append(dict(fold=held+1,hours=hrs,alerts=len(idx),incidents_grouped=ng,incidents_raw=nr,
                   alerts_per_h=len(idx)/hrs,incidents_per_h=ng/hrs,events=len(ev),
                   events_detected=cov,median_latency_s=float(np.median(lat)) if lat else np.nan))
T4u=pd.DataFrame(CU); print('\nTABLE 4 UNSUPERVISED'); print(T4u.round(3).to_string(index=False))
A2,I2,R2=T4u.alerts.sum(),T4u.incidents_grouped.sum(),T4u.incidents_raw.sum()
print('UNSUP alerts %d -> incidents %d (%.1f%%), raw %d (%.1f%% more), %.2f -> %.2f per h, events %d/%d, median latency %.0f s' % (
    A2,I2,100*(1-I2/A2),R2,100*(R2/I2-1),A2/T4u.hours.sum(),I2/T4u.hours.sum(),
    T4u.events_detected.sum(),T4u.events.sum(),np.nanmedian(T4u.median_latency_s)))
T4u.to_csv(f'{OUT}/table4u_consolidation_unsupervised.csv',index=False)

## 9. Export tables and figures

In [ ]:
for n,t in [('table1_unsupervised',T1),('table2_supervised',T2),('table3_localization',T3),
            ('table4_consolidation',T4),('table5_ablation',T5)]:
    t.to_csv(f'{OUT}/{n}.csv',index=False)

In [ ]:
# ---------- figures ----------
f1=[T1.pca_F1.mean(),T1.iso_F1.mean(),T2.F1.mean()]; au=[T1.pca_AUC.mean(),T1.iso_AUC.mean(),T2.AUC.mean()]
x=np.arange(3); fig,ax=plt.subplots(figsize=(5.2,3))
b1=ax.bar(x-0.19,f1,0.38,label='F1',color='0.35',hatch='//',edgecolor='black')
b2=ax.bar(x+0.19,au,0.38,label='ROC AUC',color='0.8',hatch='..',edgecolor='black')
for b in list(b1)+list(b2): ax.text(b.get_x()+b.get_width()/2,b.get_height()+0.015,'%.3f'%b.get_height(),ha='center',fontsize=7)
ax.set_xticks(x); ax.set_xticklabels(['PCA reconstruction\n(normal only)','Isolation Forest\n(normal only)','LightGBM\n(supervised)'])
ax.set_ylim(0,1.12); ax.legend(frameon=False,ncol=2,loc='lower right'); ax.set_ylabel('score')
ax.set_title('Mean detection performance, HAI 21.03')
fig.tight_layout(); fig.savefig(FIG+'/fig_detection.png'); plt.close(fig)

fig,ax=plt.subplots(figsize=(5.2,3)); x=np.arange(len(T4))
b1=ax.bar(x-0.19,T4.alerts_per_h,0.38,label='raw alerts',color='0.35',hatch='//',edgecolor='black')
b2=ax.bar(x+0.19,T4.incidents_per_h,0.38,label='consolidated incidents',color='0.8',hatch='..',edgecolor='black')
for b in list(b1)+list(b2): ax.text(b.get_x()+b.get_width()/2,b.get_height()+0.15,'%.2f'%b.get_height(),ha='center',fontsize=7)
ax.set_xticks(x); ax.set_xticklabels(['File %d'%f for f in T4.fold]); ax.set_ylabel('events per hour')
ax.set_ylim(0,max(T4.alerts_per_h)*1.25); ax.legend(frameon=False,ncol=2)
ax.set_title('Operator alert burden per test file')
fig.tight_layout(); fig.savefig(FIG+'/fig_alert_burden.png'); plt.close(fig)

held=1; sv=store[held]['sv']; gm=np.stack([sv[:,GROUPS==g].sum(1) for g in G],1)
o=np.argsort(Wte[held]['t'][store[held]['idx']])
fig,ax=plt.subplots(figsize=(6,2.4))
im=ax.imshow(gm[o].T,aspect='auto',cmap='RdBu_r',vmin=-np.abs(gm).max(),vmax=np.abs(gm).max())
ax.set_yticks(range(4)); ax.set_yticklabels(['P1 boiler','P2 turbine','P3 water','P4 HIL'])
ax.set_xlabel('alert window (time ordered)'); ax.set_title('Subsystem grouped SHAP attribution')
fig.colorbar(im,ax=ax,shrink=0.8); fig.tight_layout(); fig.savefig(FIG+'/fig_grouped_shap.png'); plt.close(fig)
print('\nK=%d  elapsed %ds' % (K,time.time()-t0))

In [ ]:
for n, t in [('Table 1 unsupervised', T1), ('Table 2 supervised', T2), ('Table 3 localisation', T3),
             ('Table 4 consolidation', T4u), ('Table 5 ablation', T5)]:
    print('\n' + n)
    print(t.to_latex(index=False, float_format=lambda v: '%.3f' % v))

!cd /content && zip -qr tcce_results.zip outputs figures && ls -la /content/tcce_results.zip